# Off-market · cross-family blend screen

**Goal:** test whether adding a separately trained CatBoost and/or LightGBM off-market predictor improves the frozen 7.4%-MdAPE XGBoost ensemble.

The existing frozen ensemble stays **read-only**. Its saved `shadow_audit_predictions.csv` defines the historical homes and baseline forecasts. CatBoost and LightGBM are trained only on earlier sold homes using **off-market-safe features** (no list price or observed sale price on evaluation homes).

Predeclared blends include 80/20, 70/20/10 and 60/25/15 weighted geometric averages. Pilot challengers use less compute, so prefer their paired deltas as early clues rather than definitive evidence.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / "offmarket_crossfamily_blend.py").is_file():
    raise FileNotFoundError("Open this root notebook from the repository root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from offmarket_crossfamily_blend import REFERENCE, BLENDS, run_offmarket_blend
print("Repository:",ROOT)
print("Frozen reference:",REFERENCE)
if not REFERENCE.is_file():
    print("REQUIRED: copy the historical saved shadow_audit_predictions.csv or set REFERENCE_FILE below.")
display(pd.DataFrame([
    {"blend":name, "weights":", ".join(f"{model} {w:.0%}" for model,w in weights.items())}
    for name,weights in BLENDS.items()
]))

## 1 · Quick mixed-family screen

Train CatBoost and LightGBM using up to 25,000 earlier sales and 350 trees, and evaluate up to 350 audit homes per month in August–October 2025. The reference predictions come **unchanged** from the frozen full-size XGBoost ensemble. Individual family forecasts are checkpointed by month.

This is a **screen, not equal-budget confirmation**.

In [ ]:
REFERENCE_FILE = Path(REFERENCE)
RUN_PILOT = True
PILOT_OUTPUT = Path("outputs/offmarket_crossfamily_pilot")
if RUN_PILOT:
    if not REFERENCE_FILE.is_file():
        raise FileNotFoundError(f"Missing historical ensemble audit: {REFERENCE_FILE}")
    pooled, monthly, predictions = run_offmarket_blend(
        full=False, reference_file=REFERENCE_FILE,
        output_dir=PILOT_OUTPUT, threads=4, bootstrap_repeats=300
    )
else:
    pooled = pd.read_csv(PILOT_OUTPUT/"pooled_scores.csv")
    monthly = pd.read_csv(PILOT_OUTPUT/"monthly_scores.csv")
    predictions = pd.read_csv(PILOT_OUTPUT/"predictions.csv",dtype={"ml_num":str})
display(pooled[["arm","variant","homes","mdape","mape","p90_ape",
                "delta_mdape_pp","delta_mape_pp","delta_mdape_ci_low",
                "delta_mdape_ci_high"]].round(4))

In [ ]:
import matplotlib.pyplot as plt
display(monthly.pivot_table(index="variant",columns="month",values="delta_mdape_pp").round(3))
fig,ax=plt.subplots(figsize=(10,4))
p=pooled.sort_values("delta_mdape_pp")
ax.barh(p.variant,p.delta_mdape_pp)
ax.axvline(0,color="gray")
ax.invert_yaxis()
ax.set_xlabel("Change in MdAPE vs FROZEN XGBoost ensemble (percentage points)")
plt.tight_layout()
plt.show()

## 2 · Optional matched full-history/iteration follow-up

Run this only if the pilot suggests that one or both alternative families contribute useful improvements. It trains new CatBoost and LightGBM models on *all* eligible earlier sold homes using their recorded tuned iteration budgets.

Even the full-mode historical evaluation is on already inspected August–October 2025 shadow-audit homes, so do not use its scores to adjust weights and claim independent confirmation.

In [ ]:
RUN_FULL = False
FULL_OUTPUT = Path("outputs/offmarket_crossfamily_full")
if RUN_FULL:
    full_pooled, full_monthly, _ = run_offmarket_blend(
        full=True, reference_file=REFERENCE_FILE,
        output_dir=FULL_OUTPUT, threads=4, bootstrap_repeats=500
    )
    display(full_pooled.round(4))
else:
    print("Full-history off-market cross-family fit disabled; set RUN_FULL=True to opt in.")

## Decision

Compare the frozen ensemble against `frozen_plus_cat_80_20`, `frozen_plus_lgb_80_20` and three-family blends. If they consistently reduce both MdAPE and MAPE across months at full strength, they may justify **a separate** ensemble-release candidate. Never overwrite the frozen production off-market model based on these exploratory folds.